In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load dataset
df = pd.read_csv("telecom_customer_churn.csv")

# Standardize column names
df.columns = df.columns.str.strip().str.lower()

print("First 5 rows:")
print(df.head())
print("\nDataset shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# 2. Remove duplicates and standardize text
df = df.drop_duplicates()

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip().str.lower()

# 3. Convert columns to correct data types
cols = ["seniorcitizen", "tenure", "monthlycharges", "totalcharges"]

for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 4. Remove rows with missing target
df = df.dropna(subset=["churn"])

# 5. Handle missing values
for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include="object").columns:
    if col not in ["customerid", "churn"] and df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

# 6. Handle outliers using IQR
for col in ["monthlycharges", "totalcharges"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    df[col] = df[col].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

# 7. Feature engineering
df["tenuregroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 48, float("inf")],
    labels=["new", "medium", "long"]
)

# 8. Export cleaned dataset
df.to_csv("Telecom_Customer_Churn_Cleaned.csv", index=False)

# 9. Separate features and target
X = df.drop(columns=["customerid", "churn"])
X = pd.get_dummies(X, drop_first=True, dtype=int)
y = df["churn"].map({"yes": 1, "no": 0})

# 10. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 11. Scale numerical features
scale_cols = ["tenure", "monthlycharges", "totalcharges"]
scaler = StandardScaler()

X_train = X_train.copy()
X_test = X_test.copy()

X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

# 12. Export encoded dataset
encoded = X.copy()
encoded["churn"] = y
encoded.to_csv("Telecom_Customer_Churn_Encoded.csv", index=False)

# Results
print("\nCleaned dataset shape:", df.shape)
print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)
print("\nChurn distribution:\n", df["churn"].value_counts())
print("\nMissing values after cleaning:", df.isnull().sum().sum())
print("\nBoth datasets exported successfully!")

First 5 rows:
   customerid  gender  seniorcitizen partner dependents  tenure phoneservice  \
0  7590-VHVEG     NaN              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      multiplelines internetservice onlinesecurity  ... deviceprotection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            NaN  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  techsupport streamingtv streamingmovies       